# 4IASDR — 2026
# Construire des Agents IA avec LangChain
## Chapitre 5 — Model Context Protocol (MCP) 
## Travaux Pratiques

**Objectifs pédagogiques**

A l'issue de ces travaux pratiques, vous serez capable de :
- Écrire un serveur MCP en Python avec `FastMCP` et le décorateur `@mcp.tool()`.
- Tester un serveur MCP de manière isolée avec l'inspector officiel.
- Connecter un agent LangChain à un serveur MCP via `MultiServerMCPClient`.
- Comprendre la différence entre invocation synchrone et asynchrone.
- Composer plusieurs serveurs MCP dans un seul agent.
- Ajouter un middleware de journalisation sur des outils MCP.

---

## Prérequis
Avant de commencer, assurons-nous d'avoir installé les paquets nécessaires.

In [1]:
# Installation des dépendances
!pip install mcp langchain-mcp-adapters langchain-openai python-dotenv

# Vérification de l'installation
!python -c "import mcp; print('mcp', mcp.__version__)"
!python -c "import langchain_mcp_adapters; print('adapters OK')"

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/98.6 kB ? eta -:--:--
   ---------------- ----------------------- 41.0/98.6 kB 1.9 MB/s eta 0:00:01
   ------------------------------------- -- 92.2/98.6 kB 1.0 MB/s eta 0:00:01
   ---------------------------------------- 98.6/98.6 kB 940.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   -------- ------------------------------- 112.6/548.1 kB 3.3 MB/s eta 0:00:01
   ---------- ----------------------------- 143.4/548.1 kB 2.1 MB/s eta 0:00:01
   -------------------- ------------------- 276.5/548.1 kB 2.1 MB/s eta 0:00:01
   -------------------------- ------------- 368.6/548.1 kB 2.1 MB/s eta 0:00:01
   ----------------------------------- ---- 491.5/548.1 kB 2.2 MB/s eta 0:00:01
   ---------------------------------------- 548.1/548.1 kB 2.0 MB/s eta 0:00:00
   -------------


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\khouloud\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import mcp; print('mcp', mcp.__version__)
    ^^^^^^^^^^
ModuleNotFoundError: No module named 'mcp'
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import langchain_mcp_adapters; print('adapters OK')
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'langchain_mcp_adapters'


**Configuration des variables d'environnement**
Le fichier `.env` doit contenir `OPENAI_API_KEY`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# Remplacement de secours si pas de fichier .env
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "votre_cle_api_openrouter_ici"
    print("Attention: Clé API par défaut utilisée. Assurez-vous d'avoir un fichier .env valide.")
else:
    print("Clé API chargée avec succès.")

--- 
## Exercice 1 — Premier serveur MCP (niveau 1/3)

**Contexte**
Vous allez créer un serveur MCP minimal qui expose deux outils de calcul. L'objectif est de comprendre la structure d'un serveur FastMCP et le rôle des docstrings avant d'aborder un cas métier plus complexe.

**Travail demandé : Étape 1.1 Créer le fichier `math_server.py`**
Le serveur doit exposer exactement deux outils (`add`, `multiply`) avec leurs docstrings en anglais. 
Le serveur doit se lancer en mode stdio et intégrer un `print` dans chaque outil.

**Bonus :** Outil `divide`.

In [ ]:
%%writefile math_server.py
import sys
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("MathServer")

@mcp.tool()
def add(a: float, b: float) -> float:
    """Additionne deux nombres et retourne la somme. A utiliser quand il faut faire une addition."""
    result = a + b
    print(f"[math_server] add({a}, {b}) = {result}", file=sys.stderr)
    return result

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiplie deux nombres et retourne le produit. A utiliser quand il faut faire une multiplication."""
    result = a * b
    print(f"[math_server] multiply({a}, {b}) = {result}", file=sys.stderr)
    return result

@mcp.tool()
def divide(a: float, b: float) -> str | float:
    """Divise le premier nombre par le second et retourne le quotient. Gere la division par zero."""
    if b == 0:
        error_msg = "Error: Division by zero is not allowed."
        print(f"[math_server] divide({a}, {b}) = {error_msg}", file=sys.stderr)
        return error_msg
    result = a / b
    print(f"[math_server] divide({a}, {b}) = {result}", file=sys.stderr)
    return result

if __name__ == "__main__":
    mcp.run(transport="stdio")

**Étape 1.2 Tester avec l'inspector**

Ouvrez un terminal et exécutez la commande suivante :
```bash
npx @modelcontextprotocol/inspector python math_server.py
```
Vous pourrez vérifier les appels `add`, `multiply`, et `divide` depuis l'interface web.

--- 
## Exercice 2 — Premier agent consommateur (niveau 2/3)

**Contexte**
Créer un agent LangChain qui se connecte au serveur `math_server.py`. Maîtriser `MultiServerMCPClient` et le mode asynchrone.

**Étape 2.1 à 2.4 - Fichier `math_agent.py`**
Le script suivant crée l'agent et pose la question "Calcule (17 + 23) x 4".

In [ ]:
%%writefile math_agent.py
import os
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Fonction de creation d'agent pour respecter la consigne create_agent()
def create_agent(model, tools, middleware=None):
    if middleware:
        wrapped_tools = []
        for t in tools:
            wrapped = t
            for m in middleware:
                wrapped = m(wrapped)
            wrapped_tools.append(wrapped)
    else:
        wrapped_tools = tools
        
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Tu es un assistant mathématique utile."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ])
    agent = create_tool_calling_agent(model, wrapped_tools, prompt)
    return AgentExecutor(agent=agent, tools=wrapped_tools, verbose=False)

async def main():
    # 1. Declarer un MultiServerMCPClient pointant vers math_server.py
    client = MultiServerMCPClient({
        "math": {
            "command": "python",
            "args": ["math_server.py"],
            "transport": "stdio"
        }
    })
    
    async with client:
        # 2. Appeler await client.get_tools()
        tools = await client.get_tools()
        
        # 3. Afficher dans le terminal le nombre d'outils decouverts et leurs noms
        tool_names = [t.name for t in tools]
        print(f"Outils decouverts ({len(tools)}) : {tool_names}")
        
        # Configuration du LLM (Etape 2.2)
        llm = ChatOpenAI(
            model="openai/gpt-oss-120b:free",
            openai_api_key=os.getenv("OPENAI_API_KEY", "dummy"),
            openai_api_base="https://openrouter.ai/api/v1",
        )
        
        # 4. Creer un agent LangChain
        agent_executor = create_agent(model=llm, tools=tools)
        
        # 5. Poser la question et afficher la reponse finale
        question = "Calcule (17 + 23) x 4"
        print(f"\nQuestion : {question}")
        response = await agent_executor.ainvoke({"input": question})
        print(f"Reponse finale : {response['output']}")

if __name__ == "__main__":
    asyncio.run(main())

--- 
## Exercice 3 — Multi-serveurs et middleware (niveau 3/3)

**Étape 3.1 Serveur `hr_server.py`**

In [ ]:
%%writefile hr_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("HRServer")

EMPLOYEES = {
    "E001": {"name": "Alice Dupont", "department": "IT", "salary": 60000},
    "E002": {"name": "Bob Martin", "department": "HR", "salary": 55000},
    "E003": {"name": "Karim Bennani", "department": "Sales", "salary": 70000},
}

@mcp.tool()
def get_employee(employee_id: str) -> str:
    """Get the profile of an employee by their ID."""
    employee = EMPLOYEES.get(employee_id)
    if employee:
        return f"Employee {employee_id}: {employee['name']}, Department: {employee['department']}"
    return f"Employee {employee_id} not found."

@mcp.tool()
def list_employees_by_department(department: str) -> str:
    """List all employees in a specific department."""
    emps = [f"{eid}: {e['name']}" for eid, e in EMPLOYEES.items() if e["department"] == department]
    if emps:
        return f"Employees in {department}: " + ", ".join(emps)
    return f"No employees found in department {department}."

@mcp.tool()
def update_salary(employee_id: str, new_salary: int) -> str:
    """Update the salary of an employee."""
    if employee_id in EMPLOYEES:
        EMPLOYEES[employee_id]["salary"] = new_salary
        return f"Salary of employee {employee_id} updated to {new_salary}."
    return f"Employee {employee_id} not found."

if __name__ == "__main__":
    mcp.run(transport="stdio")

**Étape 3.2 Serveur `notes_server.py`**

In [ ]:
%%writefile notes_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("NotesServer")

NOTES = {}

@mcp.tool()
def add_note(employee_id: str, note: str) -> str:
    """Attache une note textuelle au dossier de l'employe."""
    if employee_id not in NOTES:
        NOTES[employee_id] = []
    NOTES[employee_id].append(note)
    return f"Note ajoutee avec succes au dossier de l'employe {employee_id}."

@mcp.tool()
def get_notes(employee_id: str) -> list[str]:
    """Retourne la liste des notes du dossier de l'employe."""
    return NOTES.get(employee_id, [])

if __name__ == "__main__":
    mcp.run(transport="stdio")

**Étape 3.3 à 3.5 Agent `dual_agent.py`**
Avec middleware de journalisation (`logger_middleware`) et appel aux deux serveurs.

In [ ]:
%%writefile dual_agent.py
import os
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import BaseTool

def logger_middleware(tool: BaseTool) -> BaseTool:
    """Middleware de journalisation (logger_middleware)."""
    original_arun = tool._arun
    original_run = tool._run
    
    async def wrapped_arun(*args, **kwargs):
        print(f"[MCP] tool={tool.name}  args={kwargs}")
        return await original_arun(*args, **kwargs)
        
    def wrapped_run(*args, **kwargs):
        print(f"[MCP] tool={tool.name}  args={kwargs}")
        return original_run(*args, **kwargs)
        
    tool._arun = wrapped_arun
    tool._run = wrapped_run
    return tool

def create_agent(model, tools, middleware=None):
    if middleware:
        wrapped_tools = []
        for t in tools:
            wrapped = t
            for m in middleware:
                wrapped = m(wrapped)
            wrapped_tools.append(wrapped)
    else:
        wrapped_tools = tools
        
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an assistant capable of managing employee data and notes. You have access to tools from both 'hr' and 'notes' servers."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ])
    agent = create_tool_calling_agent(model, wrapped_tools, prompt)
    return AgentExecutor(agent=agent, tools=wrapped_tools, verbose=False)

async def main():
    client = MultiServerMCPClient({
        "hr": {
            "command": "python",
            "args": ["hr_server.py"],
            "transport": "stdio"
        },
        "notes": {
            "command": "python",
            "args": ["notes_server.py"],
            "transport": "stdio"
        }
    })
    
    async with client:
        tools = await client.get_tools()
        
        # Affichage decouverte
        tool_names = [t.name for t in tools]
        print(f"Outils decouverts ({len(tools)}) : {tool_names}")
        
        llm = ChatOpenAI(
            model="openai/gpt-oss-120b:free",
            openai_api_key=os.getenv("OPENAI_API_KEY", "dummy"),
            openai_api_base="https://openrouter.ai/api/v1",
        )
        
        # Passage du middleware
        agent_executor = create_agent(model=llm, tools=tools, middleware=[logger_middleware])
        
        question = "Recupere la fiche de E003 puis ajoute la note 'Excellent trimestre' a son dossier."
        print(f"\nQuestion : {question}")
        response = await agent_executor.ainvoke({"input": question})
        print(f"\nReponse finale : {response['output']}")

if __name__ == "__main__":
    asyncio.run(main())

--- 
## Exécution et Génération de `trace.txt`

Les cellules suivantes vont exécuter les agents créés et sauvegarder leurs sorties dans le fichier `trace.txt` demandé pour le rendu.

In [ ]:
import subprocess

def run_and_trace(cmd, trace_file):
    print(f"Exécution de: {cmd}")
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    output = ""
    for line in process.stdout:
        print(line, end="")
        output += line
    with open(trace_file, "a", encoding="utf-8") as f:
        f.write(f"\n--- Trace de la commande : {cmd} ---\n")
        f.write(output)

# On efface le fichier de trace précédent s'il existe
with open("trace.txt", "w", encoding="utf-8") as f:
    f.write("=== Trace d'exécution du TP MCP ===\n")

run_and_trace("python math_agent.py", "trace.txt")
print("\n" + "="*50 + "\n")
run_and_trace("python dual_agent.py", "trace.txt")

--- 
## Modalités de rendu - Création de l'archive ZIP
La cellule ci-dessous crée le fichier `TP_MCP_NomPrenom.zip` contenant l'ensemble de votre travail.

In [ ]:
import zipfile
import os

files_to_zip = [
    "math_server.py",
    "math_agent.py",
    "notes_server.py",
    "hr_server.py",
    "dual_agent.py",
    "trace.txt",
    "TP_MCP_Complet_4IASDR_2026.ipynb"  # Inclut le notebook lui-meme si sauvegarde
]

zip_filename = "TP_MCP_NomPrenom.zip"

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file)
            print(f"Ajouté : {file}")
        else:
            print(f"Attention : le fichier {file} est introuvable. N'oubliez pas de sauvegarder le notebook et de l'executer.")

print(f"\nArchive {zip_filename} générée avec succès ! Vous pouvez la télécharger pour la déposer sur la plateforme.")